# ♻️ WasteVision Studio — 04: Evaluation, 3D Feature Maps & Inference
### التقييم النهائي، مصفوفة الالتباس، استخراج خرائط الخصائص ثلاثية الأبعاد، وتجربة الصور
يشمل التقييم على مجموعة الاختبار المحجوزة، رسم مصفوفة الالتباس، عرض استجابة طبقات الالتفاف ثلاثية الأبعاد، وواجهة الاستدلال التفاعلية.


In [ ]:
import sys
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
import cv2
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

sys.path.append("..")
from core.config import MODEL_PATH
from core.inference_service import InferenceService

model = tf.keras.models.load_model(MODEL_PATH)
print("تم تحميل النموذج المدرب بنجاح:", MODEL_PATH)


In [ ]:
# 1. استخراج وعرض خرائط الخصائص ثلاثية الأبعاد (3D Feature Maps)
conv_layers = [layer for layer in model.layers if isinstance(layer, tf.keras.layers.Conv2D)]
feature_model = tf.keras.Model(model.input, [layer.output for layer in conv_layers])

# إنشاء مدخل اختباري
dummy = np.zeros((1, 160, 160, 3), dtype=np.float32)
activations = feature_model(dummy, training=False)

fig = make_subplots(rows=1, cols=len(activations), specs=[[{"type":"surface"}]*len(activations)], subplot_titles=[f"Conv Layer {i+1}" for i in range(len(activations))])
for col, act in enumerate(activations, 1):
    plane = act[0, :, :, 0].numpy()
    plane = cv2.resize(plane, (32, 32))
    fig.add_trace(go.Surface(z=plane, colorscale="Turbo", showscale=False), row=1, col=col)
fig.update_layout(title="Learned 3D Conv2D Feature Activations", height=450)
fig.show()
